# Tema 4 - Laboratorul 5: Rețele Markov
## Exercițiul 2: Putem prezice vremea la o stație fără observații?
Vom construi un model Markov Random Field (MRF) pentru predicția vremii. Considerăm $n=4$ stații meteo care formează un arbore pentru a nu avea cicluri.

### a) Desenarea grafului
Stațiile $S_1, S_2, S_3, S_4$ sunt modelate astfel încât stația 2 este conectată de celelalte trei (configurație de tip stea/arbore).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product

# Setări generale
n = 4
edges = [(0, 1), (1, 2), (1, 3)]  # Muchii (S1-S2, S2-S3, S2-S4) cu indexare de la 0
states = ["Soare", "Ploaie", "Nori"]

# Vizualizarea grafului
G = nx.Graph()
G.add_edges_from([(u+1, v+1) for u, v in edges]) # Etichete de la 1 la 4

plt.figure(figsize=(6, 4))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='#D8BFD8', node_size=1000, font_weight='bold')
plt.title("Graful stațiilor meteo (fără cicluri)")
plt.show()


### b) & c) Construirea factorilor și calculul distribuției comune
Definim factorii unari (probabilitățile inițiale $p_i(s)$) și factorii pereche ($\Psi_{s_i, s_j}$). Folosim enumerarea exhaustivă a celor $3^4 = 81$ configurații pentru a calcula constanta de normalizare $Z$ și probabilitățile comune.


In [ ]:
# Probabilități inițiale și factorul pereche
p_unary = np.array([0.5, 0.3, 0.2])
Psi = np.array([
    [0.7, 0.2, 0.1],
    [0.2, 0.6, 0.2],
    [0.1, 0.3, 0.6]
])

# Funcție pentru calculul produsului de factori dintr-o configurație
def unnormalized_prob(cfg):
    val = 1.0
    for i in range(n):
        val *= p_unary[cfg[i]]
    for (u, v) in edges:
        val *= Psi[cfg[u], cfg[v]]
    return val

# Enumerăm toate configurațiile posibile
configs = list(product([0, 1, 2], repeat=n))
unnorm_probs = np.array([unnormalized_prob(c) for c in configs])

# Constanta de normalizare Z și probabilitățile normalizate
Z = np.sum(unnorm_probs)
probs = unnorm_probs / Z

print(f"Constanta de normalizare Z calculată exact este: {Z:.5f}")


### d) & e) Distribuția marginală vs. Configurația comună MAP
Pentru a studia diferențele dintre Marginal MAP și Joint MAP, prespunem o observație: la stația $S_1$ (index 0) observăm "Ploaie" (index 1) și la $S_4$ (index 3) observăm "Nori" (index 2).


In [ ]:
# Observații: S1 = Ploaie(1), S4 = Nori(2)
obs_nodes = [0, 3]
obs_vals = [1, 2]

# Filtrăm doar configurațiile compatibile cu observațiile
mask = np.ones(len(configs), dtype=bool)
for node, val in zip(obs_nodes, obs_vals):
    mask &= np.array([cfg[node] == val for cfg in configs])

cond_probs = probs[mask] / np.sum(probs[mask])
cond_configs = [configs[i] for i in range(len(configs)) if mask[i]]

# MAP Marginal pentru stațiile neobservate
marg_map_states = []
print("--- Distribuții Marginale Condiționate ---")
for i in range(n):
    if i in obs_nodes:
        val = obs_vals[obs_nodes.index(i)]
        marg_map_states.append(states[val])
        continue
    
    marg = np.zeros(3)
    for cfg, p in zip(cond_configs, cond_probs):
        marg[cfg[i]] += p
    
    best_state = np.argmax(marg)
    marg_map_states.append(states[best_state])
    print(f"S{i+1}: Soare={marg[0]:.3f}, Ploaie={marg[1]:.3f}, Nori={marg[2]:.3f} -> MAP Marginal: {states[best_state]}")

# MAP Comun (Joint MAP)
joint_map_idx = np.argmax(cond_probs)
joint_map_cfg = cond_configs[joint_map_idx]
joint_map_states = [states[s] for s in joint_map_cfg]

print(f"\nConfigurația Marginal MAP: {marg_map_states}")
print(f"Configurația Joint MAP:    {joint_map_states}")



**Interpretare pentru Exercițiul 2:**
Cele două metode pot selecta configurații identice în cazuri simple, dar **nu coincid întotdeauna**. Alegerea separată a maximelor marginale este o decizie strict "locală", ignorând dependențele puternice impuse de potențialul pereche pe muchii. Maximizarea comună (Joint MAP) găsește starea coerentă per ansamblu, asigurându-se că valorile vecine au compatibilitate maximă conform lui $\Psi$.

---

## Exercițiul 3: Restaurarea unei imagini binare cu un MRF
Construim o imagine, o alterăm cu zgomot și încercăm să îi refacem structura optimizând o rețea Markov prin algoritmul iterativ ICM (Iterated Conditional Modes).


In [ ]:
# a) & b) Construirea imaginii și adăugarea zgomotului
img_orig = np.zeros((5, 5), dtype=int)
img_orig[1:4, 1:4] = 1  # Pătrat alb în centru

np.random.seed(42)  # Sămânță fixă pentru reproductibilitate
img_noisy = img_orig.copy()
noise_mask = np.random.rand(5, 5) < 0.1
img_noisy[noise_mask] = 1 - img_noisy[noise_mask]

# Vizualizare inițială
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img_orig, cmap='gray'); axes[0].set_title("Original")
axes[1].imshow(img_noisy, cmap='gray'); axes[1].set_title("Zgomot (y)")
plt.show()


### c) & d) Definirea energiei și implementarea ICM
Energia este definită prin $E(x;y) = \lambda \sum_i (x_i - y_i)^2 + \sum_{(i,j)} (x_i - x_j)^2$. Minimul energiei reprezintă o configurație MAP. Folosim ICM (o metodă aproximativă, iterativă, care optimizează lăcomos energia pe rând pentru fiecare pixel).


In [ ]:
def run_icm(y, lambda_val, iterations=5):
    h, w = y.shape
    x = y.copy()
    
    for _ in range(iterations):
        for i in range(h):
            for j in range(w):
                # Energie unară (fidelitate față de observație)
                e0 = lambda_val * (0 - y[i,j])**2
                e1 = lambda_val * (1 - y[i,j])**2
                
                # Energie pereche (netezire / coerență cu vecinii 4-conectați)
                for ni, nj in [(i-1, j), (i+1, j), (i, j-1), (i, j+1)]:
                    if 0 <= ni < h and 0 <= nj < w:
                        e0 += (0 - x[ni,nj])**2
                        e1 += (1 - x[ni,nj])**2
                
                # Update pixel pentru a minimiza energia locală (corespondent MAP local)
                x[i,j] = 0 if e0 < e1 else 1
    return x


### e) Evaluarea pentru diferite valori ale lui $\lambda$


In [ ]:
lambdas = [0.3, 1.0, 3.0]
results = [run_icm(img_noisy, l) for l in lambdas]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img_orig, cmap='gray'); axes[0].set_title("Original")
axes[1].imshow(img_noisy, cmap='gray'); axes[1].set_title("Noisy (y)")

for ax, l, res in zip(axes[2:], lambdas, results):
    errors = np.sum(res != img_orig)
    ax.imshow(res, cmap='gray')
    ax.set_title(f"ICM (λ={l})\nErori: {errors}")

plt.tight_layout()
plt.show()


**Interpretare pentru Exercițiul 3:**
Cele două componente ale ecuației de energie joacă roluri opuse:
*   **$\lambda$ mic (ex: 0.3):** Termenul de coeziune (netezire) domină. Algoritmul prioritizează uniformizarea regiunilor ignorând destul de mult observația originală `y`. Îndepărtează perfect zgomotul (erori puține în zone mari), dar tinde să taie din colțurile formei (prea multă netezire).
*   **$\lambda$ mare (ex: 3.0):** Termenul de fidelitate domină. Algoritmul este forțat să păstreze pixelii cât mai apropiați de observația inițială `y`, preferând să tolereze zgomotul (erori multe) în loc să netezească regiunea.
